<a href="https://colab.research.google.com/github/Sagar-Gusain/FashionMNIST/blob/main/HyperParameterTunign_using_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [2]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

In [3]:
torch.manual_seed(42)
device='cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
class CNN(nn.Module):
    def __init__(self, filter, dropout, hidden):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, filter, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(filter, filter*2, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(filter*2, filter*4, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filter*4*3*3, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform

)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 170kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.33MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.6MB/s]


In [10]:
from torch.utils.data import DataLoader, random_split
train_data, val_data = random_split(
    train_dataset,
    [50000, 10000],
    generator=torch.Generator().manual_seed(42)
)

In [11]:
def objective(trial):

    # Hyperparameter search space
    filter = trial.suggest_categorical('filter', [16, 32, 64])

    hidden = trial.suggest_categorical('hidden', [128, 256, 512, 1024])

    dropout = trial.suggest_float('dropout', 0.1, 0.5)

    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])

    # Create DataLoaders
    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_data,
        batch_size=batch_size,
        shuffle=False
    )

    # Initialize a fresh model
    model = CNN(filter, dropout, hidden).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    # Training
    for epoch in range(5):

        model.train()

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Validation
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    return accuracy

In [12]:
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=20)

[I 2026-10-31 13:07:39,834] A new study created in memory with name: no-name-4167a959-6bc4-48c6-a01a-46d147008962
[I 2026-10-31 13:08:20,951] Trial 0 finished with value: 0.8439 and parameters: {'filter': 16, 'hidden': 512, 'dropout': 0.35774605600306497, 'lr': 0.00011683223309065787, 'batch_size': 64}. Best is trial 0 with value: 0.8439.
[I 2026-10-31 13:08:57,284] Trial 1 finished with value: 0.8931 and parameters: {'filter': 16, 'hidden': 512, 'dropout': 0.31602087772894494, 'lr': 0.0006068573836026796, 'batch_size': 64}. Best is trial 1 with value: 0.8931.
[I 2026-10-31 13:09:44,492] Trial 2 finished with value: 0.9091 and parameters: {'filter': 16, 'hidden': 256, 'dropout': 0.19382219977707316, 'lr': 0.0008966339757492626, 'batch_size': 32}. Best is trial 2 with value: 0.9091.
[I 2026-10-31 13:10:31,442] Trial 3 finished with value: 0.9046 and parameters: {'filter': 16, 'hidden': 128, 'dropout': 0.4841657042262223, 'lr': 0.002170055505214039, 'batch_size': 32}. Best is trial 2 wit